# Conditional Poisson Sampling

Suppose you want to draw a random subset of exactly $n$ items from a universe of $N$, where each item has a different "importance" weight $w_i > 0$.  The probability of drawing a particular subset $S$ should be proportional to the product of its weights:

$$
P(S) \propto \prod_{i \in S} w_i, \quad |S| = n
$$

This is the **conditional Poisson distribution** (also called the *exponential* fixed-size design).  It's the unique maximum-entropy distribution over size-$n$ subsets for given marginal inclusion probabilities — the fixed-size analogue of independent Bernoulli sampling.

The normalizing constant for this distribution is

$$
Z = \sum_{S : |S|=n} \prod_{i \in S} w_i = e_n(w)
$$

where $e_n(w)$ is the $n$-th [elementary symmetric polynomial](https://en.wikipedia.org/wiki/Elementary_symmetric_polynomial) — the sum of all products of $n$ distinct weights.  For even moderate $N$, there are $\binom{N}{n}$ terms in this sum, so brute force is out of the question.

In this post, I'll describe an implementation that computes inclusion probabilities, draws exact samples, and fits weights to target probabilities — all in $O(N \log^2 n)$ time using a polynomial product tree.  The code is available as a single-file library: [`conditional_poisson.py`](https://github.com/timvieira/conditional-poisson-sampling).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pylab as pl
from conditional_poisson import ConditionalPoisson

## Basic usage

The simplest entry point is `from_weights`: hand it a subset size $n$ and a weight vector $w$.

In [ ]:
N, n = 10, 4
rng = np.random.default_rng(0)
w = rng.exponential(1.0, N)

cp = ConditionalPoisson.from_weights(n, w)
print(f'N={N}, n={n}')
print(f'weights:   {np.round(w, 3)}')
print(f'pi:        {np.round(cp.pi, 3)}')
print(f'sum(pi):   {cp.pi.sum():.6f}  (should be {n})')
print(f'log Z:     {cp.log_normalizer:.4f}')

The inclusion probabilities $\pi_i = P(i \in S)$ always sum to $n$, and each $\pi_i \in (0, 1)$.  Items with larger weights get higher inclusion probabilities.

## Sampling

Drawing samples works by walking a binary tree top-down, splitting a "quota" of $n$ items between the left and right subtrees at each node.  Each split is exact (not approximate), and the tree is built once and cached, so subsequent samples are cheap.

In [ ]:
M = 100_000
samples = cp.sample(M, rng=rng)
print(f'sample shape: {samples.shape}')   # (M, n)
print(f'first few samples:')
print(samples[:5])

Let's verify that the empirical inclusion frequencies match the exact $\pi$ values.

In [ ]:
pi_emp = np.bincount(samples.ravel(), minlength=N) / M

pl.figure(figsize=(8, 3))
ix = np.arange(N)
pl.bar(ix - 0.15, cp.pi, width=0.3, label='exact $\pi_i$', color='steelblue')
pl.bar(ix + 0.15, pi_emp, width=0.3, label=f'empirical ({M:,} samples)', color='coral')
pl.xlabel('item $i$')
pl.ylabel('inclusion probability')
pl.legend()
pl.title('Exact vs. empirical inclusion probabilities')
pl.tight_layout()

## The polynomial product tree

The key idea is to encode the sum over all $\binom{N}{n}$ subsets as the coefficient of $z^n$ in a product of polynomials:

$$
(1 + w_1 z)(1 + w_2 z) \cdots (1 + w_N z) = \sum_{k=0}^{N} e_k(w)\, z^k
$$

The $n$-th coefficient of this product is exactly $e_n(w) = Z$, the normalizing constant.  This product can be computed in $O(N \log^2 n)$ time using a binary tree — each leaf holds a degree-1 polynomial $(1 + w_i z)$, and internal nodes multiply their children's polynomials (truncated to degree $n$).

To get the inclusion probability $\pi_i$, we need the "leave-one-out" product — the product of all factors except factor $i$.  Rather than computing $N$ separate products, a top-down pass propagates "outside" polynomials from the root back to the leaves, reusing the tree.  At leaf $i$:

$$
\pi_i = w_i \cdot [z^{n-1}]\, P^{(-i)}(z) \;/\; Z
$$

where $P^{(-i)}(z) = \prod_{j \neq i}(1 + w_j z)$ is the leave-one-out polynomial and $[z^k]$ denotes "coefficient of $z^k$."

This is essentially the same divide-and-conquer structure I described in the [heaps for incremental computation](https://timvieira.github.io/blog/heaps-for-incremental-computation/) post, but over polynomials instead of scalars.

## Brute-force verification

For small instances, we can enumerate all $\binom{N}{n}$ subsets and verify that the efficient algorithm gets the right answer.

In [ ]:
from itertools import combinations

N_small, n_small = 7, 3
w_small = rng.exponential(1.0, N_small)
cp_small = ConditionalPoisson.from_weights(n_small, w_small)

# Brute force: enumerate all subsets
all_S = list(combinations(range(N_small), n_small))
log_probs_bf = np.array([np.log(np.prod(w_small[list(s)])) for s in all_S])
log_probs_bf -= np.log(np.exp(log_probs_bf).sum())   # normalise

# Inclusion probabilities from brute force
probs_bf = np.exp(log_probs_bf)
pi_bf = np.zeros(N_small)
for k, s in enumerate(all_S):
    for i in s:
        pi_bf[i] += probs_bf[k]

print(f'max |pi_tree - pi_brute_force| = {np.max(np.abs(cp_small.pi - pi_bf)):.2e}')
print(f'sum P(S) via brute force       = {probs_bf.sum():.10f}')
print(f'log Z: tree={cp_small.log_normalizer:.6f}, brute force={np.log(np.exp(log_probs_bf + cp_small.log_normalizer).sum()):.6f}')

## Fitting weights to target probabilities

A common use case: you know the inclusion probabilities you *want* (e.g., from a survey design or an optimal allocation), and you need to find weights that produce them.

This is a convex optimization problem.  We maximise the log-likelihood

$$
L(\theta) = \pi^{*\top} \theta - \log Z(\theta)
$$

where $\theta_i = \log w_i$ are the log-weights.  The gradient is $\pi^* - \pi(\theta)$ and the Hessian is $-\text{Cov}[Z]$, the negative covariance matrix of the inclusion indicators.  We use Newton-CG — the Hessian-vector product $\text{Cov}[Z]\, v$ is computed efficiently via the D-tree (a second tree that piggybacks on the P-tree).

In [ ]:
# Target: the inclusion probabilities from our earlier example
pi_star = cp.pi.copy()

# Fit from scratch
cp_fit = ConditionalPoisson.fit(pi_star, n, verbose=True)

In [ ]:
print(f'max |pi_fit - pi_target| = {np.max(np.abs(cp_fit.pi - pi_star)):.2e}')

pl.figure(figsize=(8, 3))
ix = np.arange(N)
pl.bar(ix - 0.15, pi_star, width=0.3, label='target $\pi^*_i$', color='steelblue')
pl.bar(ix + 0.15, cp_fit.pi, width=0.3, label='fitted $\pi_i$', color='coral')
pl.xlabel('item $i$')
pl.ylabel('inclusion probability')
pl.legend()
pl.title('Fitting weights to target inclusion probabilities')
pl.tight_layout()

## Numerical stability

The implementation handles extreme weight ranges without overflow.  Every polynomial in the tree is stored in a scaled representation `(coeffs, log_scale)` with $\max|c_k| = 1$, so FFT convolutions always operate on $O(1)$-magnitude numbers.  Weights are also geometrically normalised before each tree build.

Let's verify with some stress tests.

In [ ]:
cases = [
    ('large positive theta',  ConditionalPoisson(4, rng.uniform(30, 50, 10))),
    ('large negative theta',  ConditionalPoisson(4, rng.uniform(-50, -30, 10))),
    ('wide range theta',      ConditionalPoisson(4, np.linspace(-30, 30, 10))),
    ('N=100, n=50, uniform',  ConditionalPoisson(50, np.full(100, 2.0))),
    ('N=200, n=100, uniform', ConditionalPoisson(100, np.full(200, 10.0))),
    ('N=500, n=250, uniform', ConditionalPoisson(250, np.full(500, 5.0))),
]

print(f'{"case":30s}  {"sum(pi)":>10s}  {"log Z":>12s}  {"ok":>4s}')
print('-' * 62)
for name, cpt in cases:
    pi_t = cpt.pi
    ok = (np.isfinite(pi_t).all() and np.isfinite(cpt.log_normalizer)
          and abs(pi_t.sum() - cpt.n) < 1e-4)
    print(f'{name:30s}  {pi_t.sum():10.4f}  {cpt.log_normalizer:12.2f}  {"✓" if ok else "✗":>4s}')

## Timing

The tree-based approach scales to moderately large $N$ comfortably.

In [ ]:
import time

print(f'{"N":>6s}  {"n":>6s}  {"pi (ms)":>10s}  {"hvp (ms)":>10s}  {"10k samples (ms)":>18s}')
print('-' * 56)
for N_t, n_t in [(50, 20), (100, 40), (200, 80), (500, 200)]:
    w_t = rng.exponential(1.0, N_t)
    v_t = rng.standard_normal(N_t)
    cp_t = ConditionalPoisson.from_weights(n_t, w_t)

    reps = max(1, int(800 / (N_t * n_t**0.5)))

    t0 = time.perf_counter()
    for _ in range(reps): cp_t._cache.clear(); cp_t.pi
    pi_ms = (time.perf_counter() - t0) / reps * 1000

    t0 = time.perf_counter()
    for _ in range(reps): cp_t.hvp(v_t)
    hvp_ms = (time.perf_counter() - t0) / reps * 1000

    t0 = time.perf_counter()
    cp_t.sample(10_000, rng=rng)
    samp_ms = (time.perf_counter() - t0) * 1000

    print(f'{N_t:>6d}  {n_t:>6d}  {pi_ms:>10.1f}  {hvp_ms:>10.1f}  {samp_ms:>18.0f}')

## Summary

The conditional Poisson distribution is the natural maximum-entropy distribution over fixed-size subsets.  The polynomial product tree gives us an efficient way to compute everything we need:

 * **Upward pass**: builds the product polynomial $\prod_i (1 + w_i z)$ in a binary tree.  The root's $n$-th coefficient is the normalizing constant $Z = e_n(w)$.

 * **Downward pass**: propagates leave-one-out polynomials to every leaf, giving inclusion probabilities $\pi_i$ in a single $O(N \log^2 n)$ sweep.

 * **Sampling**: walks the tree top-down, splitting a quota at each node.  Each sample costs $O(n \log N)$.  The tree is built once and reused.

 * **Hessian-vector products**: a second "D-tree" piggybacks on the P-tree, enabling Newton-CG fitting to target inclusion probabilities.

All polynomials use a scaled representation that prevents floating-point overflow even for extreme weights.

Code: [`conditional_poisson.py`](https://github.com/timvieira/conditional-poisson-sampling)